In [1]:
# confusion_matrix.ipynb - plot confusion matrix.

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns
from logging import info, error
from matplotlib.patches import Patch
from sklearn.metrics import confusion_matrix

In [3]:
in_dir = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/subclone_inference/1n2t_newcna/base'
out_dir = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/subclone_inference/1n2t_newcna/analysis/confusion_matrix'

In [4]:
def assert_e(path):
    """Assert file or folder exists, mimicking shell "test -e"."""
    assert path is not None and os.path.exists(path)

def plot_labels_confusion_matrix(
    tool_list,
    tool_fn_list,
    truth_fn,
    out_fig_fn,
    fig_width = None,
    fig_height = None,
    fig_nrow = None,
    fig_ncol = 3,
    fig_dpi = 300,
    verbose = True
):
    # check args.
    assert len(tool_list) == len(tool_fn_list)
    new_tool_list, new_tool_fn_list = [], []
    for tool, fn in zip(tool_list, tool_fn_list):
        if not os.path.exists(fn):
            print("W: '%s' file does not exist!" % tool)
        else:
            new_tool_list.append(tool)
            new_tool_fn_list.append(fn)
    tool_list, tool_fn_list = new_tool_list, new_tool_fn_list
    assert_e(truth_fn)


    # plot confusion matrix for each tool.
    if verbose:
        info("plot confusion matrix for each tool's labels ...")

    n = len(tool_list)
    fig_ncol = min(n, fig_ncol)
    if fig_nrow is None:
        fig_nrow = int(np.ceil(n / fig_ncol))
    if fig_width is None:
        fig_width = 2.5 * fig_ncol
    if fig_height is None:
        fig_height = 2.5 * fig_nrow
    
    # Create subplots with exact number needed (no empty subplots)
    fig, axes = plt.subplots(
        fig_nrow, fig_ncol,
        figsize = (fig_width, fig_height)
    )
    
    # Handle axes array - flatten if 2D, convert to list if single
    if n == 1:
        axes = [axes]
    elif fig_nrow == 1:
        axes = [axes] if not isinstance(axes, np.ndarray) else axes.tolist()
    else:
        axes = axes.flatten()
    
    # Hide unused subplots
    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)

    # Publication-ready styling
    title_fontsize = 6
    label_fontsize = 6
    tick_fontsize = 6
    annot_fontsize = 6
    
    df = pd.read_csv(truth_fn, sep = '\t')
    truth_labels = df['annotation'].to_numpy()
    i = 0
    for ax, tool, tool_fn in zip(axes[:n], tool_list, tool_fn_list):
        display_name = tool
        if 'xclone' in tool.lower():
            display_name = 'XClone'
        if verbose:
            info("process %s ..." % display_name)

        df = pd.read_csv(tool_fn, sep = '\t')
        tool_labels = df['prediction'].to_numpy()
        print(len(truth_labels), len(tool_labels))
        cm = pd.crosstab(truth_labels, tool_labels)
        print(cm)
        if i == 0:
            sns.heatmap(
                cm, annot = True, fmt = 'd', cmap = 'Blues',
                xticklabels = ['C'+str(i+1) for i in range(len(np.unique(tool_labels)))],
                yticklabels = ['C'+str(i+1) for i in range(len(np.unique(truth_labels)))],
                cbar = False,
                ax = ax,
                annot_kws = {'fontsize': annot_fontsize}
            )
        else:
            sns.heatmap(
                cm, annot = True, fmt = 'd', cmap = 'Blues',
                xticklabels = ['C'+str(i+1) for i in range(len(np.unique(tool_labels)))],
                yticklabels = False,
                cbar = False,
                ax = ax,
                annot_kws = {'fontsize': annot_fontsize}
            )            
        #ax.set_title(f'{display_name}', fontsize = title_fontsize, pad = 8)
        ax.set_title(f'{display_name}', fontsize = title_fontsize)
        ax.set_xlabel('Predicted', fontsize = label_fontsize)
        if i == 0:
            ax.set_ylabel('True', fontsize = label_fontsize)
        else:
            ax.set_ylabel(None)
        ax.tick_params(labelsize = label_fontsize)
        
        ax.tick_params(
            axis = 'x',          # Target x-axis ticks
            labelsize = tick_fontsize,
            rotation = 0         # Force horizontal (critical fix)
        )
        if i == 0:
            ax.tick_params(
                axis = 'y',          # Keep y-axis ticks as-is (vertical by default)
                labelsize = tick_fontsize,
                rotation = 90
            )
        i += 1

    #plt.subplots_adjust(wspace = 0.1)
    plt.tight_layout()
    fig.savefig(out_fig_fn, dpi = fig_dpi, bbox_inches = 'tight')
    plt.close(fig)


    res = dict(
        out_fig_fn = out_fig_fn
    )
    return(res)

In [5]:
tool_list = ['InferCNV', 'CopyKAT', 'Numbat', 'XClone_cluster-baf', 'CalicoST']

plot_labels_confusion_matrix(
    tool_list = tool_list,
    tool_fn_list = [os.path.join(in_dir, \
        'subclone_inference_k2/1_overlap/overlap.%s.tsv' % x.lower())   \
        for x in tool_list],
    truth_fn = os.path.join(in_dir, 'subclone_inference_k2/1_overlap/overlap.truth.tsv'),
    out_fig_fn = os.path.join(out_dir, 'confusion_matrix.png'),
    fig_width = 4.16,
    fig_height = 1.38,
    fig_nrow = 1,
    fig_ncol = 5,
    fig_dpi = 300,
    verbose = True
)

1000 1000
col_0    1
row_0     
0      500
1      500
1000 1000
col_0  0    1
row_0        
0      1  499
1      0  500
1000 1000
col_0    0    1
row_0          
0      453   47
1        4  496
1000 1000
col_0    0    1
row_0          
0      469   31
1       21  479
1000 1000
col_0    0    1
row_0          
0        9  491
1      500    0


{'out_fig_fn': '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/subclone_inference/1n2t_newcna/analysis/confusion_matrix/confusion_matrix.png'}